# 01 - Exploratory Data Analysis: Orders

This notebook performs comprehensive exploratory data analysis on the Walmart delivery orders dataset to understand patterns and identify potential fraud indicators.

## Objectives
- Load and understand the structure of orders data from PostgreSQL
- Analyze order distribution by value, region, and time
- Identify patterns in missing items (potential fraud indicators)
- Detect anomalies and outliers
- Generate actionable insights

## Data Source
- PostgreSQL database: `walmart-delivery-fraud-detection`
- Schema: `walmart_fraud`
- Period: January 2023 - December 2023
- Region: Central Florida

In [1]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings

import sys
sys.path.insert(0, '..')

from src.database.connection import load_orders, load_orders_full, get_summary_stats, test_connection

# Settings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

# Plotly default template
import plotly.io as pio
pio.templates.default = 'plotly_white'

print('Libraries loaded successfully!')

Libraries loaded successfully!


## 1. Database Connection and Data Loading

In [2]:
# Test database connection
if test_connection():
    print('Database connection: OK')
else:
    raise Exception('Database connection failed!')

Database connection: OK


In [3]:
# Get summary statistics
stats = get_summary_stats()
print('=' * 50)
print('DATABASE SUMMARY')
print('=' * 50)
for key, value in stats.items():
    print(f'{key}: {value}')

DATABASE SUMMARY
total_orders: 10000
total_drivers: 1247
total_customers: 1239
total_products: 314
total_missing_items: 1662
sum_items_missing: 1657
total_order_value: 2833022.38
min_date: 2023-01-01
max_date: 2023-12-31


In [4]:
# Load orders data
orders = load_orders()
print(f'Orders loaded: {len(orders):,} records')
print(f'Columns: {list(orders.columns)}')
orders.head()

Orders loaded: 10,000 records
Columns: ['order_id', 'order_date', 'order_amount', 'region', 'items_delivered', 'items_missing', 'delivery_hour', 'driver_id', 'customer_id', 'created_at']


,order_id,order_date,order_amount,region,items_delivered,items_missing,delivery_hour,driver_id,customer_id,created_at
0,44dd6e53-7cf3-46cf-bd72-f011fefcd55a,2023-01-01,212.37,Kissimmee,4,0,00:19:23,WDID10385,WCID6233,2025-12-31 04:17:14.021606+00:00
1,9fe3cd7c-7240-4e24-93e8-2196a8076812,2023-01-01,326.35,Winter Park,11,0,00:52:37,WDID10719,WCID5693,2025-12-31 04:17:14.021606+00:00
2,bfc314f8-66f8-4abc-b884-f3950a9466c2,2023-01-01,406.72,Apopka,14,0,01:13:36,WDID09883,WCID5752,2025-12-31 04:17:14.021606+00:00
3,de3f8888-9975-4e00-9f0d-89689bc34981,2023-01-01,27.55,Clermont,14,0,01:26:54,WDID11067,WCID5078,2025-12-31 04:17:14.021606+00:00
4,c48d53f9-4fa8-4d6b-a906-01a4b5ef5d7b,2023-01-01,380.69,Kissimmee,1,0,01:37:33,WDID10164,WCID5817,2025-12-31 04:17:14.021606+00:00


In [5]:
# Data types
orders.dtypes

order_id                        object
order_date                      object
order_amount                   float64
region                          object
items_delivered                  int64
items_missing                    int64
delivery_hour                   object
driver_id                       object
customer_id                     object
created_at         datetime64[ns, UTC]
dtype: object

In [6]:
# Check for missing values
missing = orders.isnull().sum()
print('Missing values:')
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values')

Missing values:
No missing values


## 2. Descriptive Statistics

In [7]:
# Basic statistics
print('=' * 60)
print('ORDERS OVERVIEW')
print('=' * 60)
print(f"Total Orders: {len(orders):,}")
print(f"Date Range: {orders['order_date'].min()} to {orders['order_date'].max()}")
print(f"Total Revenue: ${orders['order_amount'].sum():,.2f}")
print(f"Average Order Value: ${orders['order_amount'].mean():,.2f}")
print(f"Median Order Value: ${orders['order_amount'].median():,.2f}")
print(f"Min Order Value: ${orders['order_amount'].min():,.2f}")
print(f"Max Order Value: ${orders['order_amount'].max():,.2f}")
print()
print('ITEMS STATISTICS')
print('-' * 40)
total_items = orders['items_delivered'].sum() + orders['items_missing'].sum()
total_missing = orders['items_missing'].sum()
print(f"Total Items Ordered: {total_items:,}")
print(f"Total Items Delivered: {orders['items_delivered'].sum():,}")
print(f"Total Items Missing: {total_missing:,}")
print(f"Overall Missing Rate: {total_missing/total_items*100:.2f}%")
print()
orders_with_missing = (orders['items_missing'] > 0).sum()
print(f"Orders with Missing Items: {orders_with_missing:,} ({orders_with_missing/len(orders)*100:.2f}%)")

ORDERS OVERVIEW
Total Orders: 10,000
Date Range: 2023-01-01 to 2023-12-31
Total Revenue: $2,833,022.38
Average Order Value: $283.30
Median Order Value: $270.53
Min Order Value: $20.08
Max Order Value: $1,386.00

ITEMS STATISTICS
----------------------------------------
Total Items Ordered: 101,351
Total Items Delivered: 99,694
Total Items Missing: 1,657
Overall Missing Rate: 1.63%

Orders with Missing Items: 1,502 (15.02%)


In [8]:
# Numeric summary
orders[['order_amount', 'items_delivered', 'items_missing']].describe()

,order_amount,items_delivered,items_missing
count,10000.00,10000.00,10000.00
mean,283.30,9.97,0.17
std,181.68,5.46,0.41
min,20.08,1.00,0.00
25%,147.56,5.00,0.00
50%,270.53,10.00,0.00
75%,393.07,15.00,0.00
max,1386.00,19.00,3.00


## 3. Order Amount Analysis

In [9]:
# Order amount distribution
fig = px.histogram(
    orders, 
    x='order_amount', 
    nbins=50,
    title='Distribution of Order Amounts',
    labels={'order_amount': 'Order Amount ($)', 'count': 'Number of Orders'},
    color_discrete_sequence=['#0071CE']
)
fig.add_vline(x=orders['order_amount'].mean(), line_dash='dash', line_color='red', 
              annotation_text=f"Mean: ${orders['order_amount'].mean():.2f}")
fig.add_vline(x=orders['order_amount'].median(), line_dash='dot', line_color='green',
              annotation_text=f"Median: ${orders['order_amount'].median():.2f}")
fig.update_layout(height=400)
fig.show()

In [10]:
# Order amount by region
fig = px.box(
    orders, 
    x='region', 
    y='order_amount',
    title='Order Amount Distribution by Region',
    labels={'order_amount': 'Order Amount ($)', 'region': 'Region'},
    color='region'
)
fig.update_layout(xaxis_tickangle=-45, showlegend=False, height=450)
fig.show()

In [11]:
# Order amount percentiles
percentiles = [10, 25, 50, 75, 90, 95, 99]
print('Order Amount Percentiles:')
for p in percentiles:
    val = orders['order_amount'].quantile(p/100)
    print(f'  {p}th percentile: ${val:,.2f}')

Order Amount Percentiles:
  10th percentile: $71.46
  25th percentile: $147.56
  50th percentile: $270.53
  75th percentile: $393.07
  90th percentile: $468.85
  95th percentile: $491.46
  99th percentile: $1,025.22


## 4. Missing Items Analysis (Fraud Indicators)

In [12]:
# Create derived features
orders['total_items'] = orders['items_delivered'] + orders['items_missing']
orders['missing_rate'] = orders['items_missing'] / orders['total_items'] * 100
orders['has_missing'] = orders['items_missing'] > 0

# Handle edge cases
orders['missing_rate'] = orders['missing_rate'].fillna(0)

In [13]:
# Missing items distribution
fig = px.histogram(
    orders, 
    x='items_missing',
    title='Distribution of Missing Items per Order',
    labels={'items_missing': 'Number of Missing Items', 'count': 'Number of Orders'},
    color_discrete_sequence=['#DC3545']
)
fig.update_layout(height=400)
fig.show()

In [14]:
# Orders with vs without missing items
missing_counts = orders['has_missing'].value_counts()
labels = ['No Missing Items', 'Has Missing Items']
colors = ['#28A745', '#DC3545']

fig = px.pie(
    values=[missing_counts.get(False, 0), missing_counts.get(True, 0)], 
    names=labels,
    title='Proportion of Orders with Missing Items',
    color_discrete_sequence=colors,
    hole=0.4
)
fig.update_traces(textposition='outside', textinfo='percent+label+value')
fig.update_layout(height=400)
fig.show()

In [15]:
# Missing rate distribution (only for orders with missing items)
orders_with_issues = orders[orders['has_missing']]

fig = px.histogram(
    orders_with_issues, 
    x='missing_rate',
    nbins=20,
    title='Missing Rate Distribution (Orders with Missing Items Only)',
    labels={'missing_rate': 'Missing Rate (%)', 'count': 'Number of Orders'},
    color_discrete_sequence=['#FFC107']
)
fig.update_layout(height=400)
fig.show()

print(f"\nOrders with 100% missing rate: {(orders['missing_rate'] == 100).sum()}")
print(f"Orders with >50% missing rate: {(orders['missing_rate'] > 50).sum()}")


Orders with 100% missing rate: 0
Orders with >50% missing rate: 5


In [16]:
# Anomalous orders: missing > delivered
anomalous = orders[orders['items_missing'] > orders['items_delivered']]
print(f'Anomalous orders (missing > delivered): {len(anomalous)}')
if len(anomalous) > 0:
    display(anomalous[['order_id', 'order_date', 'region', 'items_delivered', 'items_missing', 'order_amount']])

Anomalous orders (missing > delivered): 5


,order_id,order_date,region,items_delivered,items_missing,order_amount
1583,9e7458e7-69ac-480a-abc8-c45d208ab04c,2023-02-28,Winter Park,1,2,990.52
2617,b03c4219-b407-4de6-97b6-2a19b6aabbbd,2023-04-07,Clermont,1,2,612.64
2644,1206f2bb-cc3f-4a5e-b07e-49c08c12298b,2023-04-09,Apopka,1,2,1045.57
4495,fc1a9a63-238c-4781-8324-a4b81eb2cac9,2023-06-16,Apopka,1,2,181.20
9129,44d24056-a449-4ed6-b39a-d477f511ced8,2023-11-30,Apopka,1,2,98.29


## 5. Regional Analysis

In [17]:
# Aggregate by region
regional = orders.groupby('region').agg({
    'order_id': 'count',
    'order_amount': 'sum',
    'items_delivered': 'sum',
    'items_missing': 'sum'
}).reset_index()

regional.columns = ['region', 'total_orders', 'total_revenue', 'items_delivered', 'items_missing']
regional['total_items'] = regional['items_delivered'] + regional['items_missing']
regional['missing_rate'] = regional['items_missing'] / regional['total_items'] * 100
regional['avg_order_value'] = regional['total_revenue'] / regional['total_orders']
regional['pct_of_orders'] = regional['total_orders'] / regional['total_orders'].sum() * 100

# Sort by missing rate
regional = regional.sort_values('missing_rate', ascending=False)
regional

,region,total_orders,total_revenue,items_delivered,items_missing,total_items,missing_rate,avg_order_value,pct_of_orders
0,Altamonte Springs,1426,408562.15,14134,253,14387,1.76,286.51,14.26
1,Apopka,1422,405920.39,14153,249,14402,1.73,285.46,14.22
2,Clermont,1384,390672.57,13886,243,14129,1.72,282.28,13.84
4,Orlando,1401,396044.15,13986,233,14219,1.64,282.69,14.01
6,Winter Park,1485,419550.38,14790,235,15025,1.56,282.53,14.85
5,Sanford,1461,420210.16,14400,225,14625,1.54,287.62,14.61
3,Kissimmee,1421,392062.58,14345,219,14564,1.50,275.91,14.21


In [18]:
# Missing rate by region - horizontal bar chart
avg_missing_rate = orders['items_missing'].sum() / (orders['items_delivered'].sum() + orders['items_missing'].sum()) * 100

fig = px.bar(
    regional.sort_values('missing_rate'), 
    x='missing_rate', 
    y='region',
    orientation='h',
    title='Missing Rate by Region (Fraud Risk Indicator)',
    labels={'missing_rate': 'Missing Rate (%)', 'region': 'Region'},
    color='missing_rate',
    color_continuous_scale='Reds',
    text='missing_rate'
)
fig.add_vline(x=avg_missing_rate, line_dash='dash', line_color='blue', 
              annotation_text=f'Average: {avg_missing_rate:.2f}%')
fig.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
fig.update_layout(height=400)
fig.show()

In [19]:
# Orders volume by region
fig = px.bar(
    regional.sort_values('total_orders', ascending=False), 
    x='region', 
    y='total_orders',
    title='Order Volume by Region',
    labels={'total_orders': 'Number of Orders', 'region': 'Region'},
    color='total_revenue',
    color_continuous_scale='Blues',
    text='total_orders'
)
fig.update_traces(texttemplate='%{text:,}', textposition='outside')
fig.update_layout(xaxis_tickangle=-45, height=400)
fig.show()

In [20]:
# Identify regional hotspots (above average missing rate)
print('REGIONAL HOTSPOTS (Above Average Missing Rate)')
print('=' * 50)
hotspots = regional[regional['missing_rate'] > avg_missing_rate]
for _, row in hotspots.iterrows():
    print(f"{row['region']}: {row['missing_rate']:.2f}% missing rate ({row['items_missing']:,} items)")

REGIONAL HOTSPOTS (Above Average Missing Rate)
Altamonte Springs: 1.76% missing rate (253 items)
Apopka: 1.73% missing rate (249 items)
Clermont: 1.72% missing rate (243 items)
Orlando: 1.64% missing rate (233 items)


## 6. Temporal Analysis

In [21]:
# Convert order_date to datetime if needed
orders['order_date'] = pd.to_datetime(orders['order_date'])

# Extract temporal features
orders['month'] = orders['order_date'].dt.month
orders['month_name'] = orders['order_date'].dt.month_name()
orders['day_of_week'] = orders['order_date'].dt.day_name()
orders['day_of_week_num'] = orders['order_date'].dt.dayofweek
orders['hour'] = pd.to_datetime(orders['delivery_hour'].astype(str)).dt.hour

In [ ]:
# Monthly trends
monthly = orders.groupby('month').agg({
    'order_id': 'count',
    'order_amount': 'sum',
    'items_delivered': 'sum',
    'items_missing': 'sum'
}).reset_index()

monthly['missing_rate'] = monthly['items_missing'] / (monthly['items_delivered'] + monthly['items_missing']) * 100
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
monthly['month_label'] = monthly['month'].apply(lambda x: month_names[x-1])

# Dual axis plot
fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Bar(x=monthly['month_label'], y=monthly['order_id'], name='Orders', marker_color='#0071CE'),
    secondary_y=False
)

fig.add_trace(
    go.Scatter(x=monthly['month_label'], y=monthly['missing_rate'], name='Missing Rate %', 
               mode='lines+markers', marker_color='#DC3545', line=dict(width=3)),
    secondary_y=True
)

fig.update_layout(title='Monthly Orders and Missing Rate Trend (2023)', height=450)
fig.update_yaxes(title_text='Number of Orders', secondary_y=False)
fig.update_yaxes(title_text='Missing Rate (%)', secondary_y=True)
fig.show()

In [ ]:
# Day of week analysis
daily = orders.groupby(['day_of_week', 'day_of_week_num']).agg({
    'order_id': 'count',
    'items_delivered': 'sum',
    'items_missing': 'sum'
}).reset_index().sort_values('day_of_week_num')

daily['missing_rate'] = daily['items_missing'] / (daily['items_delivered'] + daily['items_missing']) * 100

fig = px.bar(
    daily, 
    x='day_of_week', 
    y='missing_rate',
    title='Missing Rate by Day of Week',
    labels={'missing_rate': 'Missing Rate (%)', 'day_of_week': 'Day of Week'},
    color='missing_rate',
    color_continuous_scale='Reds',
    text='missing_rate'
)
fig.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
fig.update_layout(height=400)
fig.show()

In [ ]:
# Hourly patterns
hourly = orders.groupby('hour').agg({
    'order_id': 'count',
    'items_delivered': 'sum',
    'items_missing': 'sum'
}).reset_index()

hourly['missing_rate'] = hourly['items_missing'] / (hourly['items_delivered'] + hourly['items_missing']) * 100

# Define periods
def get_period(hour):
    if 6 <= hour < 12:
        return 'Morning (6-12)'
    elif 12 <= hour < 18:
        return 'Afternoon (12-18)'
    elif 18 <= hour < 22:
        return 'Evening (18-22)'
    else:
        return 'Night (22-6)'

hourly['period'] = hourly['hour'].apply(get_period)

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Bar(x=hourly['hour'], y=hourly['order_id'], name='Orders', marker_color='#17A2B8'),
    secondary_y=False
)

fig.add_trace(
    go.Scatter(x=hourly['hour'], y=hourly['missing_rate'], name='Missing Rate %', 
               mode='lines+markers', marker_color='#DC3545', line=dict(width=3)),
    secondary_y=True
)

fig.update_layout(title='Hourly Orders and Missing Rate Pattern', height=450)
fig.update_xaxes(title_text='Hour of Day', tickmode='linear', dtick=2)
fig.update_yaxes(title_text='Number of Orders', secondary_y=False)
fig.update_yaxes(title_text='Missing Rate (%)', secondary_y=True)
fig.show()

In [ ]:
# Period analysis
orders['period'] = orders['hour'].apply(get_period)
period_stats = orders.groupby('period').agg({
    'order_id': 'count',
    'items_delivered': 'sum',
    'items_missing': 'sum'
}).reset_index()
period_stats['missing_rate'] = period_stats['items_missing'] / (period_stats['items_delivered'] + period_stats['items_missing']) * 100
period_stats = period_stats.sort_values('missing_rate', ascending=False)

print('MISSING RATE BY PERIOD')
print('=' * 50)
for _, row in period_stats.iterrows():
    print(f"{row['period']}: {row['missing_rate']:.2f}% ({row['order_id']:,} orders)")

## 7. Correlation Analysis

In [ ]:
# Correlation matrix
numeric_cols = ['order_amount', 'items_delivered', 'items_missing', 'total_items', 'missing_rate', 'hour']
corr = orders[numeric_cols].corr()

fig = px.imshow(
    corr,
    text_auto='.2f',
    title='Correlation Matrix',
    color_continuous_scale='RdBu_r',
    aspect='auto'
)
fig.update_layout(height=500)
fig.show()

In [ ]:
# Scatter: Order Amount vs Missing Items
fig = px.scatter(
    orders[orders['has_missing']], 
    x='order_amount', 
    y='items_missing',
    color='region',
    title='Order Amount vs Missing Items (Orders with Issues)',
    labels={'order_amount': 'Order Amount ($)', 'items_missing': 'Items Missing'},
    opacity=0.6
)
fig.update_layout(height=450)
fig.show()

## 8. Key Findings and Insights

In [ ]:
# Summary statistics
print('=' * 70)
print('KEY FINDINGS - ORDERS EDA')
print('=' * 70)

# Overall stats
total_orders = len(orders)
total_revenue = orders['order_amount'].sum()
avg_order = orders['order_amount'].mean()
total_items = orders['total_items'].sum()
total_missing = orders['items_missing'].sum()
overall_missing_rate = total_missing / total_items * 100
orders_with_issues = orders['has_missing'].sum()

print(f"\n1. VOLUME OVERVIEW")
print(f"   - Total Orders: {total_orders:,}")
print(f"   - Total Revenue: ${total_revenue:,.2f}")
print(f"   - Average Order Value: ${avg_order:.2f}")
print(f"   - Date Range: {orders['order_date'].min().date()} to {orders['order_date'].max().date()}")

print(f"\n2. MISSING ITEMS (FRAUD INDICATORS)")
print(f"   - Overall Missing Rate: {overall_missing_rate:.2f}%")
print(f"   - Orders with Missing Items: {orders_with_issues:,} ({orders_with_issues/total_orders*100:.2f}%)")
print(f"   - Total Items Missing: {total_missing:,}")
print(f"   - Estimated Loss (avg $10/item): ${total_missing * 10:,.2f}")

# Highest risk region
highest_risk_region = regional.iloc[0]
print(f"\n3. HIGHEST RISK REGION")
print(f"   - Region: {highest_risk_region['region']}")
print(f"   - Missing Rate: {highest_risk_region['missing_rate']:.2f}%")
print(f"   - Items Missing: {highest_risk_region['items_missing']:,.0f}")

# Peak fraud period
peak_month = monthly.loc[monthly['missing_rate'].idxmax()]
peak_hour = hourly.loc[hourly['missing_rate'].idxmax()]
print(f"\n4. TEMPORAL PATTERNS")
print(f"   - Highest Risk Month: {peak_month['month_label']} ({peak_month['missing_rate']:.2f}%)")
print(f"   - Highest Risk Hour: {peak_hour['hour']:02d}:00 ({peak_hour['missing_rate']:.2f}%)")

print(f"\n5. ANOMALIES DETECTED")
print(f"   - Orders with missing > delivered: {len(anomalous)}")
print(f"   - Orders with 100% missing rate: {(orders['missing_rate'] == 100).sum()}")

## Summary

### Key Insights from Orders EDA:

1. **Overall Fraud Exposure**: ~15% of orders have missing items, representing significant potential fraud

2. **Regional Hotspots**: Certain regions show higher missing rates than average - these should be prioritized for investigation

3. **Temporal Patterns**: 
   - Specific months and hours show elevated missing rates
   - This could indicate organized fraud or operational issues during peak times

4. **Anomalous Orders**: Cases where missing items exceed delivered items are extreme fraud indicators

### Next Steps:
- Analyze driver and customer patterns (Notebook 02)
- Deep dive into fraud patterns (Notebook 03)
- Build ML models for fraud detection (Notebook 04)